# 02 — Bronze Auto Loader
**market-pulse-pipeline · Stage 2**

Substitui o `02_bronze_to_ingestion` (batch manual, overwrite total) por ingestão incremental automática usando **Auto Loader** (`cloudFiles`).

---

## O problema que resolve

O notebook anterior (`02_bronze_to_ingestion`) lia **todos** os ficheiros JSON em cada execução e fazia overwrite da tabela Delta. Com o tempo:

| Execuções | Ficheiros relidos | Impacto |
|---|---|---|
| 10 ingestões | 60 ficheiros | Lento |
| 100 ingestões | 600 ficheiros | Muito lento |
| 365 dias | 2190 ficheiros | Inaceitável |

O Auto Loader resolve isto com **checkpoint** — regista o que já processou e na próxima execução processa **só os ficheiros novos**.

---

## Arquitectura

```
01_bronze_ingestion (manual ou Azure Function)
    → chama Alpha Vantage API
    → escreve JSON em /bronze/stocks/ingest_date=YYYY-MM-DD/

            ↓ ficheiros novos detectados

Auto Loader (este notebook)
    → lê só os ficheiros novos (cloudFiles)
    → achata o JSON não-flat (foreachBatch + explode)
    → append à tabela Delta /bronze/ingestion/stocks/

            ↓

Silver layer (03_silver_imperative ou DLT)
    → lê de /bronze/ingestion/stocks/
    → cast de tipos, deduplicação, validação
```

---

## Porquê `foreachBatch`?

O JSON da Alpha Vantage tem uma estrutura não-flat — as datas são **chaves dinâmicas** no objecto `Time Series (Daily)`:

```json
{
  "Meta Data": { "2. Symbol": "AAPL", ... },
  "Time Series (Daily)": {
    "2026-05-12": { "1. open": "292.56", ... },
    "2026-05-11": { "1. open": "291.97", ... }
  }
}
```

O Spark Structured Streaming tem restrições em transformações aninhadas complexas (campos com espaços, números, structs dentro de Maps). O `foreachBatch` resolve isto — cada micro-batch é processado como um **DataFrame batch normal**, sem restrições.

```
Stream puro          → restrições em nested fields complexos
foreachBatch         → batch DataFrame completo, sem restrições ✅
```

---

## Schema explícito

O Auto Loader não consegue inferir automaticamente o schema deste JSON porque as datas são chaves (não valores). Definimos o schema manualmente:

```
json_schema
├── Meta Data (StructType)
│   ├── 2. Symbol          → StringType
│   ├── 3. Last Refreshed  → StringType  (cast para DateType no Silver)
│   └── 5. Time Zone       → StringType
│
└── Time Series (Daily) (MapType)
    ├── key   → StringType  (a data: "2026-05-12")
    └── value → StructType
                ├── 1. open   → StringType  (cast para DoubleType no Silver)
                ├── 2. high   → StringType
                ├── 3. low    → StringType
                ├── 4. close  → StringType
                └── 5. volume → StringType  (cast para LongType no Silver)
```

**Nota:** Todos os campos chegam como `StringType` do JSON. Os casts para tipos numéricos e de data são feitos na camada Silver — princípio de separação de responsabilidades.

---

## Transformação: de Map para linhas

O `explode` converte o Map de datas em linhas individuais:

```
ANTES (1 linha por ficheiro):
  AAPL | {"2026-05-12": {open:292,...}, "2026-05-11": {open:291,...}, ...}

DEPOIS (1 linha por data):
  AAPL | 2026-05-12 | 292.56 | 295.27 | 292.56 | 294.80 | 45748129
  AAPL | 2026-05-11 | 291.97 | 293.88 | 290.23 | 292.68 | 42247285
  ...
```

---

## Checkpoint

O checkpoint é uma pasta no ADLS onde o Auto Loader guarda o registo de todos os ficheiros já processados:

```
/bronze/_checkpoints/bronze_autoloader/
    ├── commits/        ← transacções confirmadas
    ├── offsets/        ← posição actual do stream
    └── sources/        ← ficheiros já processados
```

**Nunca apagar o checkpoint** em produção — perdes o histórico de processamento e o Auto Loader reprocessa tudo desde o início (duplicados).

---

## Trigger: `availableNow=True`

```python
.trigger(availableNow=True)
```

Processa todos os ficheiros novos disponíveis no momento e **pára**. Ideal para execução agendada (Lakeflow Jobs, cron). Alternativas:

| Trigger | Comportamento | Usar quando |
|---|---|---|
| `availableNow=True` | Processa novos e pára | Jobs agendados ✅ |
| `processingTime="5 minutes"` | Corre continuamente, micro-batch cada 5min | Streaming quase-real-time |
| `once=True` | Legacy, equivalente ao availableNow | Evitar — deprecated |

---

## Migração do sistema antigo

Na primeira execução após substituir o `02_bronze_to_ingestion`:

1. **Limpar o TARGET_PATH** (remove dados do schema antigo)
2. **Correr o Auto Loader** — processa todos os ficheiros históricos existentes
3. **A partir daí** — só ficheiros novos são processados

```python
# Limpeza única de migração — só fazer uma vez
dbutils.fs.rm(TARGET_PATH, recurse=True)
```

---

## Como usar

### Execução manual (desenvolvimento)
1. Correr `01_bronze_ingestion` para depositar novos JSONs no ADLS
2. Correr este notebook — processa os novos ficheiros e pára

### Execução automática (produção — Stage 2)
Integrado no **Lakeflow Job** como task:
```
Lakeflow Job (trigger: schedule ou evento)
    Task 1: 01_bronze_ingestion  (chama API)
    Task 2: 02_bronze_autoloader (este notebook)
    Task 3: Silver pipeline
    Task 4: Gold pipeline
```

---

## Ficheiros e paths

| Variável | Path |
|---|---|
| `LANDING_PATH` | `abfss://bronze@marketpulsedatalake.dfs.core.windows.net/stocks/` |
| `CHECKPOINT` | `abfss://bronze@marketpulsedatalake.dfs.core.windows.net/_checkpoints/bronze_autoloader/` |
| `TARGET_PATH` | `abfss://bronze@marketpulsedatalake.dfs.core.windows.net/ingestion/stocks/` |

---

## Evolução técnica

| | Stage 1 — `02_bronze_to_ingestion` | Stage 2 — `02_bronze_autoloader` |
|---|---|---|
| Leitura | `spark.read` (batch estático) | `spark.readStream` (Auto Loader) |
| Ficheiros | Relê todos sempre | Só os novos (checkpoint) |
| Escrita | `overwrite` | `append` |
| Transformação | `parse_file()` Python puro | `foreachBatch` + `explode` Spark nativo |
| Escala | ❌ degrada com o tempo | ✅ constante independentemente do histórico |
| Execução | Manual | Pronta para orquestração (Lakeflow Jobs) |

In [0]:
# Configuração
STORAGE_ACCOUNT = "marketpulsedatalake"

LANDING_PATH = f"abfss://bronze@{STORAGE_ACCOUNT}.dfs.core.windows.net/stocks/"
CHECKPOINT   = f"abfss://bronze@{STORAGE_ACCOUNT}.dfs.core.windows.net/_checkpoints/bronze_autoloader/"
TARGET_PATH  = f"abfss://bronze@{STORAGE_ACCOUNT}.dfs.core.windows.net/ingestion/stocks/"

print("✅ Configuração carregada")
print(f"  Landing:     {LANDING_PATH}")
print(f"  Checkpoint:  {CHECKPOINT}")
print(f"  Target:      {TARGET_PATH}")

In [0]:
from pyspark.sql.types import *

meta_schema = StructType([
    StructField("2. Symbol",StringType(), True),
    StructField("3. Last Refreshed",StringType(), True),
    StructField("5. Time Zone",StringType(), True)
])

time_series_value_schema = StructType([
    StructField("1. open", StringType(), True),
    StructField("2. high", StringType(), True),
    StructField("3. low", StringType(), True),
    StructField("4. close", StringType(), True),
    StructField("5. volume", StringType(), True)
])



json_schema = StructType([
    StructField("Meta Data", meta_schema, True),
    StructField("Time Series (Daily)", MapType(StringType(), time_series_value_schema), True)
])


print("✅ Schema defined")

In [0]:
df_raw = (spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("cloudFiles.schemaLocation", CHECKPOINT)
    .schema(json_schema)
    .load(LANDING_PATH)
)

In [0]:
from pyspark.sql.functions import explode, col

def process_batch(batch_df, batch_id):
    df_exploded = batch_df.selectExpr(
        "`Meta Data`.`2. Symbol` as symbol",
        "`Meta Data`.`3. Last Refreshed` as last_refreshed",
        "`Meta Data`.`5. Time Zone` as time_zone",
        "explode(`Time Series (Daily)`) as (trade_date, prices)"
    )
    df_flat = df_exploded.select(
        "symbol",
        "last_refreshed",
        "time_zone",
        "trade_date",
        col("prices.`1. open`").alias("open"),
        col("prices.`2. high`").alias("high"),
        col("prices.`3. low`").alias("low"),
        col("prices.`4. close`").alias("close"),
        col("prices.`5. volume`").alias("volume")
    )
    df_flat.write.format("delta").mode("append").save(TARGET_PATH)

In [0]:
# # Test the previous cell
# test_df = (spark.read
#     .format("json")
#     .schema(json_schema)
#     .load("abfss://bronze@marketpulsedatalake.dfs.core.windows.net/stocks/ingest_date=2026-05-13/AAPL_20260513_133143.json")
# )

# process_batch(test_df, batch_id=0)

In [0]:
(spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("cloudFiles.schemaLocation", CHECKPOINT)
    .schema(json_schema)
    .load(LANDING_PATH)
    .writeStream
    .foreachBatch(process_batch)
    .option("checkpointLocation", CHECKPOINT)
    .trigger(availableNow=True)
    .start()
)

In [0]:
from pyspark.sql.functions import countDistinct

df = spark.read.format("delta").load(TARGET_PATH)

df.agg(
    countDistinct("symbol").alias("stocks"),
    countDistinct("trade_date").alias("datas_unicas")
).display()


spark.read.format("delta").load(TARGET_PATH).display()

In [0]:
files = dbutils.fs.ls("abfss://bronze@marketpulsedatalake.dfs.core.windows.net/stocks/")
print(len(files), "ingestões")